# Liquidity Absorption 05b — Deribit millisecond trade anatomy + archive validation

**Read-only validation** of the historical Deribit trade-tape archive, followed by a **cross-venue event-time study** of BTC-PERPETUAL / ETH-PERPETUAL executions aligned to existing A05a first-touch and second-touch **one-minute candle close** times.

- Source: `history/trades/{BTC,ETH}/future/YYYY/YYYY-MM-DD.parquet`. The file combines dated and perpetual futures. We select exact `{BTC,ETH}-PERPETUAL` for the event study.
- Validation and microanalysis read the files in bounded Parquet batches; the notebook never redownloads data or reconstructs L2.
- All 452 candidate signals are included, labeled by the *existing* TARGET / STOP / TIMEOUT / NO_STRUCTURAL_TARGET / CENSORED outcome. The outcome is **retrospective descriptive metadata**, never an input to a model.
- Sparse bars at **1, 10, 100, and 1,000 milliseconds** contain only actual prints: no executions are invented in empty milliseconds. Day-boundary and stale reference problems are explicit.
- **Timestamp caveat:** A05a first/second-touch timestamps are **one-minute candle closes**, not the exact millisecond when the level was touched. Deribit perpetual market prices are not Binance USDT-M futures prices. These plots do **not** represent the identical order-book event or account P&L.
- Deribit `direction` is the exchange-reported *taker side*; this is a trade execution measure, not resting bid/ask orders, depth, or refill.

The full Jan-2024-to-now archive validator is an independent CLI task; this notebook audits the **days around A05a events** before replaying the short micro-windows. Complete archive QA can take substantial disk/CPU time.

In [2]:
from pathlib import Path
import os, sys, subprocess, json
import pandas as pd
from IPython.display import display, Image
REPO=Path.cwd().resolve()
while REPO!=REPO.parent and not (REPO/'src'/'data_engineering').is_dir(): REPO=REPO.parent
assert (REPO/'src'/'data_engineering').is_dir(), 'Start Jupyter from the repository'
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a05b import VERSION,Config,load_anchors,needed_days,build_results
from tools.validate_deribit_history_trades import VERSION as QA_VERSION,archive_root,inventory
assert VERSION=='LA-A05b-0.1.0' and QA_VERSION=='DERIBIT-HIST-VALIDATOR-0.1.0','git pull, restart kernel'
for pattern in ('test_deribit_history_validator.py','test_liquidity_absorption_stage_a05b.py'):
    p=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p',pattern,'-q'],cwd=REPO,capture_output=True,text=True)
    print(pattern,p.stdout[-500:],p.stderr[-2000:]);assert p.returncode==0,'A05b regression test failure'

root=os.environ.get('DERIBIT_DATA_ROOT')
if not root and (REPO/'.env').exists():
    for line in (REPO/'.env').read_text(encoding='utf-8').splitlines():
        if line.strip().startswith('DERIBIT_DATA_ROOT='):
            root=line.split('=',1)[1].strip().strip('"').strip("'")
assert root, 'Set DERIBIT_DATA_ROOT, or specify local history folder in the next cell'
HISTORY=archive_root(Path(root))
OUT=REPO/'notebooks'/'Liquidity_Absorption'/'05b'
print('Module:',VERSION,
'Archive:',HISTORY,
'Results:',OUT)
assert (HISTORY/'trades').is_dir(), 'Could not locate Deribit history/trades folder'


test_deribit_history_validator.py  ----------------------------------------------------------------------
Ran 12 tests in 0.290s

OK

test_liquidity_absorption_stage_a05b.py  ----------------------------------------------------------------------
Ran 16 tests in 4.263s

OK

Module: LA-A05b-0.1.0 Archive: D:\MarketData\Deribit\history Results: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\05b


## 1. Validate local historical files before interpreting prints

Only the event-related July–September 2026 UTC day files are checked in this notebook, not every day downloaded since 2024. Expect a heavy scan of ~90 days × two currencies; status **`PASS_WITH_LIMITATIONS` does not certify remote completeness**. `WARN` flags suspicious exchange-sequence spans, absence of a perpetual, large same-millisecond bursts or empty days. `MISSING_FILE` may simply indicate a download that has not completed; the downloader only publishes completed `.parquet` files. Avoid redownloading on the basis of a warning alone.

For the *full archive* after the downloader finishes:

```bat
python tools\validate_deribit_history_trades.py --currencies BTC ETH --kind future --start 2024-01-01 --end 2026-10-19 --out notebooks\Liquidity_Absorption\05b\validator_all_history
```

Use the real completed end date. `--exact-ids` is optional and memory-intensive.

In [4]:
anchors=load_anchors(REPO)
wanted=needed_days(anchors)
event_days=sorted({day for _,day in wanted})
print('Original events:',anchors.a02_candidate_row.nunique(), 'UTC anchor windows:',len(anchors),
      'distinct required UTC days:',len(event_days))
quality,manifest=inventory(HISTORY,currencies=('BTC','ETH'),kind='future',days=event_days)
validator_out=OUT/'validator'
validator_out.mkdir(parents=True,exist_ok=True)
quality.to_csv(validator_out/'deribit_daily_qc.csv',index=False)
(validator_out/'deribit_daily_qc.json').write_text(json.dumps(manifest,indent=2,default=str) + ,encoding='utf-8')
print('Validator status counts (local QA only):')
display(quality.groupby(['currency','status']).size().rename('n_days').unstack(fill_value=0))
print('Suspicious milliseconds (at least 1000 observed rows in one ms):',int(quality.get('millisecond_page_limit_buckets',pd.Series(dtype=float)).fillna(0).sum()))
print('Possible per-instrument trade_seq span gaps:',int(quality.get('possible_missing_trade_seq_span',pd.Series(dtype=float)).fillna(0).sum()))
print('Cross-day trade-sequence anomalies:',len(manifest['cross_day_sequence_anomalies']))
problems=quality[quality.status!='PASS_WITH_LIMITATIONS']
display(problems.head(25))
if any(quality.status.isin(['FAIL','UNREADABLE'])):
    raise RuntimeError('Fatal Deribit archive integrity failures; inspect validator/deribit_daily_qc.csv before running A05b')
if any(quality.status.isin(['MISSING_FILE','IN_PROGRESS_TMP'])):
    print('WARNING: Missing/in-progress days; A05b marks incomplete anchor windows and excludes them from grouped results.')


SyntaxError: invalid syntax (373322707.py, line 10)

## 2. Analyze actual Deribit executed trades around the first and second touch

The study independently reopens needed days, keeps only BTC-PERPETUAL/ETH-PERPETUAL prints within ±60 seconds, and aggregates **occupied** 1ms, 10ms, 100ms, and 1000ms buckets. File scans are batched; sparse bucket output is streamed to gzip rather than holding all prints in memory. The notebook leaves existing A02/A03d/A04/A05a files unchanged.

The six fixed phase buckets are `[-60,-10), [-10,-1), [-1,0), [0,1), [1,10), [10,60)` seconds. Blank phases represent **no observed executions**, not zero signed price change.

In [ ]:
report=build_results(REPO,HISTORY,OUT,Config())
display(report)
coverage=pd.read_csv(OUT/'05b_anchor_coverage.csv')
phase=pd.read_csv(OUT/'05b_phase_summary.csv')
files=pd.read_csv(OUT/'05b_day_scan_status.csv')
print('Anchor coverage:')
display(coverage.groupby(['anchor','symbol','status']).size().rename('n').unstack(fill_value=0))
print('Daily tape availability:')
display(files.groupby(['currency','status']).size().rename('days').unstack(fill_value=0))
print('Median phase comparisons (retrospective outcome labels):')
display(phase.round(4))


## 3. Visualize millisecond reactions and trade density

The plots compare executed-price movement and taker side at the Deribit exchange. They **cannot** establish the precise point of Binance liquidity absorption. `05b_signal_sparse_1ms_examples.png` shows only milliseconds containing executions, with marker size proportional to prints per millisecond. Those example event IDs are deterministic earliest-ID cases, not representative samples.

In [ ]:
for name in ('05b_first_touch_millisecond_phase_reactions.png',
             '05b_signal_millisecond_phase_reactions.png',
             '05b_signal_sparse_1ms_examples.png'):
    p=OUT/name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))


## 4. Interpretation / integrity gate

**Keep these questions separate:** (a) did the archive contain all remote executions? (b) does Deribit perpetual trade imbalance or 1–1000ms price change around the **Binance candle close** differ by eventual outcome? (c) do observed differences survive target geometry, clock, volatility, side, level-family and venue confounding? The first cannot be fully certified from saved files, the second is descriptive, and the third requires a later independent study.

No tick continuity should be invented across gaps; no ex-ante alpha, fill quality, latency edge, or actual L2 absorption is claimed. If an archive day is missing, the anchor's coverage is flagged and its phase rows excluded from grouped comparisons. Full raw sparse bucket CSV may be large; do not commit it to GitHub if it exceeds host limits. Commit notebook + summary files + QA report, storing oversized event buckets in your existing data archive instead.